# Binary Shape Classification — Hackathon Solution
### Dual-stream ensemble with pseudo-label refinement
- **Visual stream** : standard RGB images → ResNet-34
- **Gradient stream** : edge feature maps → ResNet-34
- Semi-supervised loop: confident negatives pseudo-labeled from test set
- BN test-time adaptation + TTA inference + probability-space ensemble
- Decision threshold tuned on held-out validation via F1

## Cell 1 — Imports

In [ ]:
import os
import glob
import copy
import json
from dataclasses import dataclass
from typing import List, Tuple, Optional

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy.special import expit as sigmoid
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
from torch.optim.swa_utils import AveragedModel, update_bn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on: {DEVICE}')

## Cell 2 — Experiment Configuration
All hyper-parameters in one place. `data_root` auto-detects Kaggle vs local environment.

In [ ]:
def _detect_data_root() -> str:
    """Return the correct dataset root for the current environment."""
    kaggle_path = '/kaggle/input/competitions/iith-deep-learning-2026-hackathon'
    if os.path.isdir(kaggle_path):
        return kaggle_path
    # fallback — search for a folder that contains both train/ and test/
    for candidate in ['/kaggle/input', '/workspace', '.']:
        if os.path.isdir(os.path.join(candidate, 'train')) or \
           os.path.isdir(os.path.join(candidate, 'test')):
            return candidate
    return '/workspace'


@dataclass
class RunConfig:
    """Central config — all tunable hyper-parameters in one place."""

    # I/O
    data_root:   str   = ''
    output_file: str   = 'submission_v9.csv'

    # Image
    img_size:    int   = 192
    batch_size:  int   = 96

    # Class imbalance
    pos_weight:  float = 0.6

    # Training schedule — Visual stream
    visual_epochs:     int = 60
    visual_swa_window: int = 15

    # Training schedule — Gradient stream
    gradient_epochs:     int = 40
    gradient_swa_window: int = 10

    # Optimiser
    peak_lr: float = 1e-3

    # Pseudo-label confidence gate
    pseudo_neg_threshold: float = 0.15

    # Validation split
    val_fraction: float = 0.15
    random_seed:  int   = 42


CFG = RunConfig(data_root=_detect_data_root())
print(json.dumps(CFG.__dict__, indent=2))

## Cell 3 — Data Registry
Scans the dataset root and builds path/label lists for train and test splits.

In [ ]:
class DataRegistry:
    """
    Scans a dataset directory and exposes:
      .train_paths, .train_labels  — labeled training images
      .test_paths                  — unlabeled test images
    """

    def __init__(self, root: str):
        self.root = root
        self.train_paths:  List[str] = []
        self.train_labels: List[int] = []
        self.test_paths:   List[str] = []
        self._scan()

    def _locate_split(self, split_name: str) -> str:
        """Handle both flat and nested directory layouts."""
        nested = os.path.join(self.root, split_name, split_name)
        return nested if os.path.isdir(nested) else os.path.join(self.root, split_name)

    def _collect_labeled(self, split_dir: str):
        for cls in (0, 1):
            cls_dir = os.path.join(split_dir, str(cls))
            for p in sorted(glob.glob(os.path.join(cls_dir, '*.png'))):
                self.train_paths.append(p)
                self.train_labels.append(cls)

    def _scan(self):
        self._collect_labeled(self._locate_split('train'))
        test_dir = self._locate_split('test')
        self.test_paths = sorted(glob.glob(os.path.join(test_dir, '*.png')))

    def summary(self):
        n_pos = sum(self.train_labels)
        n_neg = len(self.train_labels) - n_pos
        print(f'Train : {len(self.train_paths)} images  ({n_neg} neg / {n_pos} pos)')
        print(f'Test  : {len(self.test_paths)} images')
        if len(self.train_paths) == 0:
            print(f'[WARNING] No training images found under: {self.root}')
            print('  Please update CFG.data_root to point at the correct dataset folder.')


registry = DataRegistry(CFG.data_root)
registry.summary()

## Cell 4 — Train / Validation Split

In [ ]:
def make_stratified_split(
    paths:    List[str],
    labels:   List[int],
    val_frac: float = 0.15,
    seed:     int   = 42,
) -> Tuple[List, List, List, List]:
    """Return (train_paths, val_paths, train_labels, val_labels)."""
    if len(paths) == 0:
        raise RuntimeError(
            'No training images found. '
            f'Check CFG.data_root = "{CFG.data_root}" points to the correct folder.'
        )
    return train_test_split(
        paths, labels,
        test_size=val_frac,
        stratify=labels,
        random_state=seed,
    )


tr_paths, vl_paths, tr_labels, vl_labels = make_stratified_split(
    registry.train_paths,
    registry.train_labels,
    val_frac=CFG.val_fraction,
    seed=CFG.random_seed,
)
print(f'Train split : {len(tr_paths)}  |  Val split : {len(vl_paths)}')

## Cell 5 — Gradient Feature Extractor
Converts RGB images into 3-channel edge maps: Canny edges, Sobel magnitude, Sobel phase.

In [ ]:
class GradientFeatureExtractor:
    """
    Callable transform: PIL image → 3-channel gradient feature map.
    Ch-0: Canny binary edges
    Ch-1: Sobel gradient magnitude
    Ch-2: Sobel gradient phase / direction
    """

    def __init__(
        self,
        bilateral_d:     int   = 9,
        bilateral_sigma: float = 150.0,
        canny_low:       int   = 30,
        canny_high:      int   = 100,
    ):
        self.bd = bilateral_d
        self.bs = bilateral_sigma
        self.cl = canny_low
        self.ch = canny_high

    @staticmethod
    def _safe_norm(arr: np.ndarray) -> np.ndarray:
        mx = arr.max()
        return (arr / mx).astype(np.float32) if mx > 0 else arr.astype(np.float32)

    def __call__(self, pil_img: Image.Image) -> Image.Image:
        grey     = np.array(pil_img.convert('L'))
        smoothed = cv2.bilateralFilter(grey, self.bd, self.bs, self.bs)

        canny_ch = cv2.Canny(smoothed, self.cl, self.ch).astype(np.float32) / 255.0

        gx    = cv2.Sobel(smoothed, cv2.CV_64F, 1, 0, ksize=3)
        gy    = cv2.Sobel(smoothed, cv2.CV_64F, 0, 1, ksize=3)
        mag_ch   = self._safe_norm(np.sqrt(gx ** 2 + gy ** 2))
        phase_ch = (cv2.phase(gx, gy, angleInDegrees=True) / 360.0).astype(np.float32)

        stacked = np.stack([canny_ch, mag_ch, phase_ch], axis=2)
        return Image.fromarray((stacked * 255).astype(np.uint8))

## Cell 6 — Dataset & MixUp Augmenter

In [ ]:
class ImageDataset(Dataset):
    """
    Unified dataset for labeled and unlabeled splits.
    When `labels` is None, __getitem__ returns only the image tensor.
    """

    def __init__(
        self,
        img_paths:  List[str],
        labels:     Optional[List[int]] = None,
        spatial_tf  = None,
        pre_tf      = None,
    ):
        self.img_paths  = img_paths
        self.labels     = labels
        self.spatial_tf = spatial_tf
        self.pre_tf     = pre_tf

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx: int):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        if self.pre_tf     is not None: img = self.pre_tf(img)
        if self.spatial_tf is not None: img = self.spatial_tf(img)
        if self.labels     is not None: return img, self.labels[idx]
        return img


class MixupAugmenter:
    """MixUp augmentation. Produces soft labels for BCEWithLogitsLoss."""

    def __init__(self, alpha: float = 0.2, enabled: bool = True):
        self.alpha   = alpha
        self.enabled = enabled

    def __call__(self, imgs: torch.Tensor, labels: torch.Tensor):
        if not self.enabled:
            return imgs, labels
        lam  = np.random.beta(self.alpha, self.alpha) if self.alpha > 0 else 1.0
        perm = torch.randperm(imgs.size(0), device=imgs.device)
        return (
            lam * imgs   + (1 - lam) * imgs[perm],
            lam * labels + (1 - lam) * labels[perm],
        )

## Cell 7 — Transform Factories

In [ ]:
_NORM_MEAN = [0.5, 0.5, 0.5]
_NORM_STD  = [0.5, 0.5, 0.5]


def build_train_transform(img_size: int, stream: str = 'visual') -> transforms.Compose:
    """Return augmentation pipeline for 'visual' or 'gradient' stream."""
    base = [
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
    ]
    if stream == 'visual':
        base += [
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomGrayscale(p=0.2),
        ]
    else:  # gradient stream — geometry only, no colour jitter
        base += [transforms.RandomRotation(15)]
    base += [
        transforms.ToTensor(),
        transforms.Normalize(_NORM_MEAN, _NORM_STD),
    ]
    return transforms.Compose(base)


def build_eval_transform(img_size: int) -> transforms.Compose:
    """Deterministic transform for validation and inference."""
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(_NORM_MEAN, _NORM_STD),
    ])


def build_tta_transforms(img_size: int) -> List[transforms.Compose]:
    """Four deterministic TTA views: identity, H-flip, V-flip, HV-flip."""
    specs = [{}, {'hflip': True}, {'vflip': True}, {'hflip': True, 'vflip': True}]
    result = []
    for spec in specs:
        ops = [transforms.Resize((img_size, img_size))]
        if spec.get('hflip'): ops.append(transforms.RandomHorizontalFlip(p=1.0))
        if spec.get('vflip'): ops.append(transforms.RandomVerticalFlip(p=1.0))
        ops += [transforms.ToTensor(), transforms.Normalize(_NORM_MEAN, _NORM_STD)]
        result.append(transforms.Compose(ops))
    return result

## Cell 8 — Model Builder

In [ ]:
def build_classifier(device: torch.device) -> nn.Module:
    """ResNet-34 from scratch with a single binary output logit."""
    backbone    = tvm.resnet34(weights=None)
    backbone.fc = nn.Linear(backbone.fc.in_features, 1)
    return backbone.to(device)

## Cell 9 — ModelTrainer

In [ ]:
class ModelTrainer:
    """
    Training engine for one model stream.
    stream_tag : 'visual' or 'gradient'
    """

    def __init__(
        self,
        model:      nn.Module,
        train_ldr:  DataLoader,
        val_ldr:    DataLoader,
        cfg:        RunConfig,
        stream_tag: str            = 'visual',
        n_epochs:   int            = 60,
        swa_window: Optional[int]  = None,
        use_mixup:  bool           = False,
    ):
        self.model     = model
        self.train_ldr = train_ldr
        self.val_ldr   = val_ldr
        self.cfg       = cfg
        self.tag       = stream_tag
        self.n_epochs  = n_epochs
        self.swa_start = (n_epochs - swa_window + 1) if swa_window else None
        self.augmenter = MixupAugmenter(alpha=0.2, enabled=use_mixup)
        self.device    = next(model.parameters()).device

        pw = torch.tensor([cfg.pos_weight], device=self.device)
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
        self.optimiser = torch.optim.AdamW(
            model.parameters(), lr=cfg.peak_lr, weight_decay=1e-2
        )
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimiser,
            max_lr=cfg.peak_lr,
            steps_per_epoch=len(train_ldr),
            epochs=n_epochs,
            pct_start=0.1,
        )
        self.best_acc   = 0.0
        self.best_state = None
        self.patience   = 0
        self._swa_model = None

    def _run_train_epoch(self):
        self.model.train()
        for imgs, lbls in self.train_ldr:
            imgs, lbls = imgs.to(self.device), lbls.float().to(self.device)
            imgs, lbls = self.augmenter(imgs, lbls)
            self.optimiser.zero_grad()
            self.criterion(self.model(imgs).squeeze(1), lbls).backward()
            self.optimiser.step()
            self.scheduler.step()

    def _run_eval_epoch(self) -> float:
        self.model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, lbls in self.val_ldr:
                logits  = self.model(imgs.to(self.device)).squeeze(1)
                correct += ((logits > 0).long() == lbls.to(self.device).long()).sum().item()
                total   += len(lbls)
        return correct / total

    def _update_swa(self, epoch: int):
        if self.swa_start is None or epoch < self.swa_start:
            return
        if self._swa_model is None:
            self._swa_model = AveragedModel(self.model)
        else:
            self._swa_model.update_parameters(self.model)

    def _load_best_weights(self):
        if self._swa_model is not None:
            update_bn(self.train_ldr, self._swa_model, device=self.device)
            cleaned = {
                k.replace('module.', ''): v
                for k, v in self._swa_model.state_dict().items()
                if 'n_averaged' not in k
            }
            self.model.load_state_dict(cleaned)
        else:
            self.model.load_state_dict(self.best_state)

    def fit(self, patience_limit: int = 8) -> nn.Module:
        for epoch in range(1, self.n_epochs + 1):
            self._run_train_epoch()
            self._update_swa(epoch)
            val_acc = self._run_eval_epoch()

            marker = ''
            if val_acc > self.best_acc:
                self.best_acc   = val_acc
                self.best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                self.patience   = 0
                marker = ' ***'
            else:
                self.patience += 1

            if epoch % 5 == 0 or marker:
                print(f'  [{self.tag}] Epoch {epoch:2d}/{self.n_epochs}  val_acc={val_acc:.4f}{marker}')

            if self.patience >= patience_limit:
                print(f'  [{self.tag}] Early stop at epoch {epoch}  (best={self.best_acc:.4f})')
                break

        self._load_best_weights()
        print(f'  [{self.tag}] Best val accuracy: {self.best_acc:.4f}')
        return self.model

## Cell 10 — DataLoader Factory

In [ ]:
def build_loaders(
    train_paths:  List[str],
    train_labels: List[int],
    val_paths:    List[str],
    val_labels:   List[int],
    cfg:          RunConfig,
    stream:       str = 'visual',
) -> Tuple[DataLoader, DataLoader, Optional[GradientFeatureExtractor], transforms.Compose]:
    """
    Returns (train_loader, val_loader, pre_tf, eval_tf).
    pre_tf is None for the visual stream; GradientFeatureExtractor for gradient stream.
    """
    pre_tf   = GradientFeatureExtractor() if stream == 'gradient' else None
    train_tf = build_train_transform(cfg.img_size, stream=stream)
    eval_tf  = build_eval_transform(cfg.img_size)
    pin      = DEVICE.type == 'cuda'

    tr_ds  = ImageDataset(train_paths, train_labels, spatial_tf=train_tf, pre_tf=pre_tf)
    vl_ds  = ImageDataset(val_paths,   val_labels,   spatial_tf=eval_tf,  pre_tf=pre_tf)
    tr_ldr = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=4, pin_memory=pin)
    vl_ldr = DataLoader(vl_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=4, pin_memory=pin)
    return tr_ldr, vl_ldr, pre_tf, eval_tf

## Cell 11 — BN Test-Time Adaptation

In [ ]:
def bn_adapt(
    model:   nn.Module,
    paths:   List[str],
    eval_tf: transforms.Compose,
    pre_tf,
    cfg:     RunConfig,
) -> None:
    """Reset and repopulate BN running stats from the given image set."""
    ds     = ImageDataset(paths, labels=None, spatial_tf=eval_tf, pre_tf=pre_tf)
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                        num_workers=4, pin_memory=(DEVICE.type == 'cuda'))
    bn_layers = [m for m in model.modules() if isinstance(m, nn.BatchNorm2d)]
    for layer in bn_layers:
        layer.train()
        layer.reset_running_stats()
    with torch.no_grad():
        for imgs in loader:
            model(imgs.to(DEVICE))
    for layer in bn_layers:
        layer.eval()

## Cell 12 — TTA Inference Engine

In [ ]:
def tta_predict(
    model:   nn.Module,
    paths:   List[str],
    eval_tf: transforms.Compose,
    pre_tf,
    cfg:     RunConfig,
) -> np.ndarray:
    """Return averaged logits over 4 TTA views as a 1-D numpy array."""
    tta_tfs     = build_tta_transforms(cfg.img_size)
    model.eval()
    accumulated = np.zeros(len(paths), dtype=np.float64)
    pin         = DEVICE.type == 'cuda'

    for aug_tf in tta_tfs:
        ds     = ImageDataset(paths, labels=None, spatial_tf=aug_tf, pre_tf=pre_tf)
        ldr    = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                            num_workers=4, pin_memory=pin)
        cursor = 0
        with torch.no_grad():
            for imgs in ldr:
                batch_logits = model(imgs.to(DEVICE)).squeeze(1).cpu().numpy()
                accumulated[cursor: cursor + len(batch_logits)] += batch_logits
                cursor += len(batch_logits)

    return accumulated / len(tta_tfs)

## Cell 13 — Pseudo-Label Selection

In [ ]:
def select_confident_negatives(
    visual_probs:   np.ndarray,
    gradient_probs: np.ndarray,
    test_paths:     List[str],
    gate:           float = 0.15,
) -> Tuple[List[str], List[int]]:
    """
    Return (paths, labels=0) for samples where both streams predict
    probability < gate simultaneously.
    """
    both_low = (visual_probs < gate) & (gradient_probs < gate)
    chosen   = np.where(both_low)[0]
    print(f'  Pseudo-label negatives selected (gate={gate}): {len(chosen)}')
    return [test_paths[i] for i in chosen], [0] * len(chosen)

## Cell 14 — Decision Threshold Tuning

In [ ]:
def tune_decision_threshold(
    true_labels: List[int],
    pred_probs:  np.ndarray,
    sweep:       np.ndarray = np.arange(0.10, 0.90, 0.05),
) -> Tuple[float, float]:
    """Grid-search threshold that maximises F1 on the validation set."""
    best_thr, best_f1 = 0.5, 0.0
    for thr in sweep:
        score = f1_score(true_labels, (pred_probs > thr).astype(int))
        if score > best_f1:
            best_f1, best_thr = score, thr
    return float(best_thr), float(best_f1)

## Cell 15 — Stream Training Helper

In [ ]:
def train_stream(
    train_paths:  List[str],
    train_labels: List[int],
    val_paths:    List[str],
    val_labels:   List[int],
    cfg:          RunConfig,
    stream:       str,
    n_epochs:     int,
    swa_window:   int,
    use_mixup:    bool,
):
    """Wire loaders + model + trainer, run fit(), return (model, eval_tf, pre_tf)."""
    tr_ldr, vl_ldr, pre_tf, eval_tf = build_loaders(
        train_paths, train_labels, val_paths, val_labels, cfg, stream=stream
    )
    model   = build_classifier(DEVICE)
    trainer = ModelTrainer(
        model, tr_ldr, vl_ldr, cfg,
        stream_tag=stream,
        n_epochs=n_epochs,
        swa_window=swa_window,
        use_mixup=use_mixup,
    )
    trained = trainer.fit(patience_limit=n_epochs)  # patience == n_epochs disables early stop
    return trained, eval_tf, pre_tf

## Cell 16 — Round 1: Train on Labeled Data

In [ ]:
print('=== Round 1 : Visual stream ===')
visual_model_r1, visual_eval_tf, visual_pre_tf = train_stream(
    tr_paths, tr_labels, vl_paths, vl_labels, CFG,
    stream='visual',
    n_epochs=CFG.visual_epochs,
    swa_window=CFG.visual_swa_window,
    use_mixup=True,
)

print('\n=== Round 1 : Gradient stream ===')
gradient_model_r1, gradient_eval_tf, gradient_pre_tf = train_stream(
    tr_paths, tr_labels, vl_paths, vl_labels, CFG,
    stream='gradient',
    n_epochs=CFG.gradient_epochs,
    swa_window=CFG.gradient_swa_window,
    use_mixup=False,
)

## Cell 17 — Pseudo-Label Generation

In [ ]:
print('=== Generating pseudo-labels ===')

visual_probe   = copy.deepcopy(visual_model_r1)
gradient_probe = copy.deepcopy(gradient_model_r1)

bn_adapt(visual_probe,   registry.test_paths, visual_eval_tf,   visual_pre_tf,   CFG)
bn_adapt(gradient_probe, registry.test_paths, gradient_eval_tf, gradient_pre_tf, CFG)

visual_logits_r1   = tta_predict(visual_probe,   registry.test_paths, visual_eval_tf,   visual_pre_tf,   CFG)
gradient_logits_r1 = tta_predict(gradient_probe, registry.test_paths, gradient_eval_tf, gradient_pre_tf, CFG)

visual_probs_r1   = sigmoid(visual_logits_r1)
gradient_probs_r1 = sigmoid(gradient_logits_r1)

pseudo_paths, pseudo_labels = select_confident_negatives(
    visual_probs_r1, gradient_probs_r1,
    registry.test_paths,
    gate=CFG.pseudo_neg_threshold,
)

del visual_probe, gradient_probe
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

aug_train_paths  = tr_paths  + pseudo_paths
aug_train_labels = tr_labels + pseudo_labels
print(f'Augmented train set: {len(aug_train_paths)} images')

## Cell 18 — Round 2: Retrain with Pseudo-Labels

In [ ]:
print('=== Round 2 : Visual stream (with pseudo-labels) ===')
visual_model, visual_eval_tf, visual_pre_tf = train_stream(
    aug_train_paths, aug_train_labels, vl_paths, vl_labels, CFG,
    stream='visual',
    n_epochs=CFG.visual_epochs,
    swa_window=CFG.visual_swa_window,
    use_mixup=True,
)

print('\n=== Round 2 : Gradient stream (with pseudo-labels) ===')
gradient_model, gradient_eval_tf, gradient_pre_tf = train_stream(
    aug_train_paths, aug_train_labels, vl_paths, vl_labels, CFG,
    stream='gradient',
    n_epochs=CFG.gradient_epochs,
    swa_window=CFG.gradient_swa_window,
    use_mixup=False,
)

## Cell 19 — Threshold Tuning on Validation Set

In [ ]:
print('=== Threshold tuning on validation set ===')

val_visual_logits   = tta_predict(visual_model,   vl_paths, visual_eval_tf,   visual_pre_tf,   CFG)
val_gradient_logits = tta_predict(gradient_model, vl_paths, gradient_eval_tf, gradient_pre_tf, CFG)

val_ensemble_probs = 0.5 * sigmoid(val_visual_logits) + 0.5 * sigmoid(val_gradient_logits)
best_threshold, best_f1 = tune_decision_threshold(vl_labels, val_ensemble_probs)
print(f'Optimal threshold : {best_threshold:.2f}   Val F1 : {best_f1:.4f}')

## Cell 20 — BN Adaptation → TTA Inference → Submission

In [ ]:
print('=== BN adaptation to test distribution ===')
bn_adapt(visual_model,   registry.test_paths, visual_eval_tf,   visual_pre_tf,   CFG)
bn_adapt(gradient_model, registry.test_paths, gradient_eval_tf, gradient_pre_tf, CFG)

print('=== Final TTA inference ===')
final_visual_logits   = tta_predict(visual_model,   registry.test_paths, visual_eval_tf,   visual_pre_tf,   CFG)
final_gradient_logits = tta_predict(gradient_model, registry.test_paths, gradient_eval_tf, gradient_pre_tf, CFG)

final_visual_probs   = sigmoid(final_visual_logits)
final_gradient_probs = sigmoid(final_gradient_logits)
ensemble_probs       = 0.5 * final_visual_probs + 0.5 * final_gradient_probs

predictions   = (ensemble_probs > best_threshold).astype(int)
file_ids      = [os.path.basename(p) for p in registry.test_paths]
submission_df = pd.DataFrame({'ID': file_ids, 'Label': predictions})
submission_df.to_csv(CFG.output_file, index=False)

print(f'Saved : {CFG.output_file}')
print(f'Positive predictions  : {predictions.sum()} / {len(predictions)}')
print(f'Visual   positives    : {(final_visual_probs   > best_threshold).sum()}')
print(f'Gradient positives    : {(final_gradient_probs > best_threshold).sum()}')